# Block 05: Linear-AE Bridge and Gauge Handling

**Goal**  
Compare EMPCA and the exact weighted-SVD bridge baseline on the same controlled rank-k family.

**Checklist alignment**  
Implements E2.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
out = run_block05_bridge_suite(cfg)
display(out["bridge_df"])

In [ ]:
bridge = out["bridge_df"].copy()
bridge["min_principal_cosine"] = bridge["principal_angle_cosines"].map(min)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bridge["k"], bridge["min_principal_cosine"], marker="o")
ax.set_xlabel("rank k")
ax.set_ylabel("minimum principal-angle cosine")
ax.set_ylim(0.0, 1.01)
ax.set_title("EMPCA vs exact weighted bridge")
save_figure(fig, dirs["figures"] / "block_05_bridge_cosines.png")
plt.close(fig)

In [ ]:
bridge_path = dirs["tables"] / "block_05_e2_bridge.csv"
manifest_path = dirs["manifests"] / "block_05_bridge_summary.json"
save_dataframe(out["bridge_df"], bridge_path)
save_json({"rows": out["bridge_df"].to_dict(orient="records")}, manifest_path)
pd.DataFrame([manifest_row("block_05", "theorem-support", str(bridge_path.relative_to(REPO_ROOT)), cfg, out["bundle"])])